In [1]:
import pandas as pd
import os

In [2]:
import pandas as pd
import os

column_names = ['Country', 'ZipCode', 'City', 'State', 'StateAbbr', 'County', 'CountyCode', 
                'Empty1', 'Empty2', 'Latitude', 'Longitude', 'Accuracy']

df = pd.read_csv('input_data1/Zip Code to GeoID.txt', sep='\t', header=None, names=column_names)

chicago_df = df[(df['StateAbbr'] == 'IL') & 
                (df['City'].str.contains('Chicago', case=False, na=False))][['ZipCode', 'Longitude', 'Latitude']]

chicago_df.to_csv('clean_data2/chicago_il_zipcodes.csv', index=False)

print(f"Chicago ZIP codes file created: {chicago_df.shape[0]} records")


Chicago ZIP codes file created: 91 records


In [3]:
housing_data = pd.read_csv('input_data1/Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv')

filtered_housing = housing_data[housing_data['City'].isin(['Chicago', 'Chicago Heights', 'Chicago Ridge'])]

date_columns = [col for col in housing_data.columns if col.startswith('20')]

columns_to_keep = ['RegionName', 'SizeRank', 'State'] + date_columns
filtered_housing = filtered_housing[columns_to_keep]

filtered_housing = filtered_housing.rename(columns={'RegionName': 'Zip Code'})

filtered_housing.to_csv('clean_data2/filtered_chicago_housing.csv', index=False)

print(f"Filtered housing data saved: {filtered_housing.shape[0]} records")

Filtered housing data saved: 58 records


In [4]:
import pandas as pd
from scipy.spatial import cKDTree
import numpy as np

# Load the crime and ZIP code datasets
crime_data = pd.read_csv('clean_data2/filtered_chicago_crimes.csv')
zip_code_data = pd.read_csv('clean_data2/chicago_il_zipcodes.csv')

# Filter crime data for the year 2023
crime_data_2023 = crime_data[crime_data['Year'] == 2023].copy()

# Build a KD-Tree for efficient nearest-neighbor search
zip_coords = zip_code_data[['Latitude', 'Longitude']].values
tree = cKDTree(zip_coords)

# Create function to assign ZIP codes to crimes based on their coordinates
def assign_zipcode_to_crimes(crime_df, chunk_size=10000):
    # Create a copy with a new ZipCode column
    crime_df = crime_df.copy()
    crime_df['ZipCode'] = None
    
    # Process in chunks to avoid memory issues
    for start_idx in range(0, len(crime_df), chunk_size):
        end_idx = min(start_idx + chunk_size, len(crime_df))
        chunk = crime_df.iloc[start_idx:end_idx].copy()
        
        # Only process rows with valid coordinates
        mask = (~chunk['Latitude'].isna()) & (~chunk['Longitude'].isna())
        valid_chunk = chunk.loc[mask]
        
        if len(valid_chunk) > 0:
            # Query the KD-Tree for nearest ZIP code
            distances, indices = tree.query(valid_chunk[['Latitude', 'Longitude']].values, k=1)
            # Assign ZIP codes using .loc to avoid SettingWithCopyWarning
            crime_df.loc[valid_chunk.index, 'ZipCode'] = zip_code_data.iloc[indices]['ZipCode'].values
            
    return crime_df

# Assign ZIP codes to crimes
crime_data_with_zip = assign_zipcode_to_crimes(crime_data_2023)

# Convert ZIP codes to string for consistent matching
crime_data_with_zip['ZipCode'] = crime_data_with_zip['ZipCode'].astype(str)
zip_code_data['ZipCode'] = zip_code_data['ZipCode'].astype(str)

# Merge crime data with ZIP code data
merged_crime_zip = pd.merge(
    zip_code_data,
    crime_data_with_zip,
    on='ZipCode',
    how='inner'
)

# Save the merged dataset
merged_crime_zip.to_csv('clean_data2/merged_crime_zip.csv', index=False)

print(f"Merged crime and ZIP code data saved: {merged_crime_zip.shape[0]} records")


Merged crime and ZIP code data saved: 261246 records


In [5]:
file_path = 'input_data1/Chicago_Public_SchoolsSY2223.csv'
schools_df = pd.read_csv(file_path)

columns_to_keep = [
    'Long_Name', 'Primary_Category', 'Zip', 'Student_Count_Total', 
    'Student_Count_Low_Income', 'Student_Count_Special_Ed', 'Student_Count_English_Learners', 
    'Student_Count_Black', 'Student_Count_Hispanic', 'Student_Count_White', 
    'Student_Count_Asian', 'Student_Count_Native_American', 'Student_Count_Other_Ethnicity', 
    'Student_Count_Asian_Pacific_Islander', 'Student_Count_Multi', 
    'Student_Count_Hawaiian_Pacific_Islander', 'Student_Count_Ethnicity_Not_Available', 
    'Overall_Rating', 'Rating_Status', 'School_Latitude', 'School_Longitude', 'Location'
]

cleaned_schools_df = schools_df[columns_to_keep]

cleaned_schools_df.loc[:, 'Year'] = 2023

cleaned_schools_df.to_csv('clean_data2/cleaned_chicago_schools.csv', index=False)

/var/folders/h6/655zg1890sg4l1s1rk51_cz00000gn/T/ipykernel_44822/3797911711.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_schools_df.loc[:, 'Year'] = 2023


In [6]:
import pandas as pd
import os

# Load all datasets
zip_code_df = pd.read_csv('clean_data2/chicago_il_zipcodes.csv')
housing_df = pd.read_csv('clean_data2/filtered_chicago_housing.csv')
crime_df = pd.read_csv('clean_data2/merged_crime_zip.csv')
schools_df = pd.read_csv('clean_data2/cleaned_chicago_schools.csv')

# Convert ZIP codes to strings for consistent matching
zip_code_df['ZipCode'] = zip_code_df['ZipCode'].astype(str)
housing_df['Zip Code'] = housing_df['Zip Code'].astype(str)
schools_df['Zip'] = schools_df['Zip'].astype(str)

# First merge: ZIP codes with housing data
merged_df = pd.merge(
    zip_code_df,
    housing_df,
    left_on='ZipCode',
    right_on='Zip Code',
    how='inner'
)

# For crime data, first filter for 2023
crime_df_2023 = crime_df[crime_df['Year'] == 2023].copy()

# Find the correct ZIP code column in crime data
crime_zip_col = None
possible_zip_columns = ['ZipCode', 'Zip', 'ZIP', 'zip', 'zip_code', 'ZIP_CODE', 'zipcode']
for col in possible_zip_columns:
    if col in crime_df_2023.columns:
        crime_zip_col = col
        break

if crime_zip_col:
    crime_df_2023[crime_zip_col] = crime_df_2023[crime_zip_col].astype(str)
    
    # Get all crime types (not just battery/assault)
    # Create dummy columns for each crime type
    crime_dummies = pd.get_dummies(crime_df_2023['Primary Type']).add_prefix('Crime_')
    crime_with_dummies = pd.concat([crime_df_2023[[crime_zip_col]], crime_dummies], axis=1)
    
    # Aggregate by ZIP code
    crime_by_zip = crime_with_dummies.groupby(crime_zip_col).sum().reset_index()
    
    # Calculate total crimes
    crime_by_zip['Total_Crimes'] = crime_by_zip.drop(crime_zip_col, axis=1).sum(axis=1)
    
    # Merge with main dataframe
    merged_df = pd.merge(
        merged_df,
        crime_by_zip,
        left_on='ZipCode',
        right_on=crime_zip_col,
        how='left'
    )
    
    # Fill NaN values with 0 for all crime columns
    crime_cols = [col for col in merged_df.columns if col.startswith('Crime_') or col == 'Total_Crimes']
    merged_df[crime_cols] = merged_df[crime_cols].fillna(0)
    
    # Get most common crime type for each ZIP code
    most_common = crime_df_2023.groupby([crime_zip_col, 'Primary Type']).size() \
                             .groupby(level=0).idxmax().apply(lambda x: x[1]).reset_index()
    most_common.columns = [crime_zip_col, 'Most_Common_Crime']
    
    merged_df = pd.merge(
        merged_df,
        most_common,
        left_on='ZipCode',
        right_on=crime_zip_col,
        how='left'
    )
else:
    print("No ZIP code column found in crime data - skipping crime data merge")

# Merge with schools data
if 'Zip' in schools_df.columns:
    merged_df = pd.merge(
        merged_df,
        schools_df,
        left_on='ZipCode',
        right_on='Zip',
        how='left'
    )

# Create output directory if it doesn't exist
output_dir = 'merged_output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save the final merged dataset
merged_df.to_csv(os.path.join(output_dir, 'final_merged_dataset.csv'), index=False)

print("Merging completed successfully!")
print(f"Final dataset includes {len(merged_df)} rows")
print("Crime data includes:")
print("- Separate columns for each crime type (Crime_[TYPE])")
print("- Total crimes count (Total_Crimes)")
print("- Most common crime type (Most_Common_Crime)")

Merging completed successfully!
Final dataset includes 657 rows
Crime data includes:
- Separate columns for each crime type (Crime_[TYPE])
- Total crimes count (Total_Crimes)
- Most common crime type (Most_Common_Crime)
